In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from spleaf import cov, term
from scipy.optimize import fmin_l_bfgs_b
import emcee
import corner

import multiGP_functions as mf

In [ ]:
STAR_PRESETS = {
    # Values match the working recipe in multiGP_run.ipynb (verified against its actual
    # output). sig_fixed=1.0: rot.sig is excluded from the fit and held at 1.0 throughout
    # optimisation + MCMC (only set to np.var(y_full) at the very end, for the final plot).
    # phi_init/k_init/gamma_1_init/delta_1_init are non-zero, non-trivial starting points --
    # important because gamma_0/gamma_1 are bounded at 0.0 on one side, so an all-zero start
    # sits exactly on that boundary and L-BFGS-B can get stuck there.
    "Sun": dict(
        data_path="../../data/Solar_Data",
        star_name="Sun",
        prot=27.0, Q=1.2,
        prot_min=20.0, prot_max=32.0, Q_min=0.9, Q_max=3.0,
        sig_fixed=1.0,
        alpha0_beta0_mult=10,
        pcyc_init=6000.0, phi_init=2.26145548, k_init=0.28573475,
        pcyc_min=3000.0, pcyc_max=6500.0,
        gamma_0_init=0.01, gamma_1_init=1.6, delta_0_init=-0.01, delta_1_init=5.0,
        delta_0_bound=(-1.0, 1.0), delta_1_bound=(-20.0, 20.0),
        planet_p_min=1.0, planet_p_max=20.0,
        planet_A_min=0.001, planet_A_max=0.5,
        planet_B_min=0.001, planet_B_max=0.5,
    ),
    # HD4628's source notebook (hd4628_fit.ipynb) uses a completely different, hand-rolled
    # negloglike (its own planet K/phi convention, not mf.negloglike's A/B) rather than
    # multiGP_functions.py -- so unlike Sun above, this preset has NOT been verified to
    # reproduce hd4628_fit.ipynb's actual results. sig_fixed=None here matches what that
    # notebook does (sig = np.var(y_full)) but everything else is a generic carry-over.
    "HD4628": dict(
        data_path="../../data/HD4628",
        star_name="HD4628",
        prot=38.0, Q=2.0,
        prot_min=36.0, prot_max=42.0, Q_min=0.5, Q_max=4.0,
        sig_fixed=None,
        alpha0_beta0_mult=5,
        pcyc_init=4800.0, phi_init=-np.pi / 6, k_init=0.1,
        pcyc_min=4000.0, pcyc_max=6000.0,
        gamma_0_init=0.0, gamma_1_init=0.0, delta_0_init=0.0, delta_1_init=0.0,
        delta_0_bound=None, delta_1_bound=None,
        planet_p_min=1.0, planet_p_max=100.0,
        planet_A_min=0.001, planet_A_max=0.1,
        planet_B_min=0.001, planet_B_max=0.1,
    ),
}

STAR = "Sun"          # "Sun" or "HD4628"
FIT_PLANET = False    # include a planet term in the fit

cfg = STAR_PRESETS[STAR]

In [ ]:
t_full, y_full, yerr_full, series_index = mf.load_and_norm_data(
    cfg["data_path"], cfg["star_name"], normalise=False, inject_planet=False
)
rv_std = 1.0

plt.figure(figsize=(12, 5))
plt.errorbar(t_full[series_index[0]], y_full[series_index[0]], yerr_full[series_index[0]], fmt='.', color='k', label='RV')
plt.legend()
plt.figure(figsize=(12, 5))
plt.errorbar(t_full[series_index[1]], y_full[series_index[1]], yerr_full[series_index[1]], fmt='.', color='k', label='RHK')
plt.legend()

In [ ]:
class MultiSeriesKernel(term.MultiSeriesKernel):
    def _grad_param(self, grad_dU=None, grad_dV=None):
        if grad_dU is not None or grad_dV is not None:
            raise NotImplementedError()
        return super()._grad_param()

In [ ]:
stds = [np.std(y_full[series_index[0]]), np.std(y_full[series_index[1]])]
sig = cfg["sig_fixed"] if cfg["sig_fixed"] is not None else np.var(y_full)

rvjit = 0.1 * stds[0]
rhkjit = 0.1 * stds[1]

C = cov.Cov(
    t_full,
    err=term.Error(yerr_full),
    rv_jit=term.InstrumentJitter(series_index[0], rvjit),
    rhk_jit=term.InstrumentJitter(series_index[1], rhkjit),
    rot=MultiSeriesKernel(
        term.SHOKernel(sig, cfg["prot"], cfg["Q"]), series_index,
        np.array([stds[0], stds[1]]),
        np.array([stds[0], 0.0]),
    ),
)

fc = mf.alpha(t_full[series_index[0]], cfg["pcyc_init"], cfg["phi_init"], cfg["k_init"])
drawn_sample = C.sample()[series_index[0]]
plt.figure()
plt.plot(t_full[series_index[0]], drawn_sample, ".", color="red", label="sample from covariance")
plt.plot(t_full[series_index[0]], fc, ".", color="blue", label="modulation function")
plt.legend()

In [ ]:
rvjit_max = 5 * stds[0]
rhkjit_max = 5 * stds[1]
alpha_0_max = cfg["alpha0_beta0_mult"] * stds[0]
alpha_1_max = 5 * stds[1]
beta_0_max = cfg["alpha0_beta0_mult"] * stds[0]
gamma_0_max = 5 * stds[0]
gamma_1_max = 5 * stds[1]
delta_0_bound = cfg["delta_0_bound"] if cfg["delta_0_bound"] is not None else (-gamma_0_max, gamma_0_max)
delta_1_bound = cfg["delta_1_bound"] if cfg["delta_1_bound"] is not None else (-gamma_1_max, gamma_1_max)

bounds_list = [
    (0.0, rvjit_max),                                  # rv_jit.sig
    (0.0, rhkjit_max),                                  # rhk_jit.sig
    (cfg["prot_min"], cfg["prot_max"]),                 # rot.P0
    (cfg["Q_min"], cfg["Q_max"]),                       # rot.Q
    (0.0, alpha_0_max),                                 # rot.alpha_0
    (-alpha_1_max, alpha_1_max),                        # rot.alpha_1
    (-beta_0_max, beta_0_max),                          # rot.beta_0
    (cfg["pcyc_min"], cfg["pcyc_max"]),                 # pcyc
    (0.0, 2 * np.pi),                                    # phi
    (-0.4, 0.4),                                        # k
    (0.0, gamma_0_max),                                 # gamma_0
    (0.0, gamma_1_max),                                 # gamma_1
    delta_0_bound,                                      # delta_0
    delta_1_bound,                                      # delta_1
]

if FIT_PLANET:
    bounds_list += [
        (cfg["planet_p_min"], cfg["planet_p_max"]),     # planet period
        (cfg["planet_A_min"], cfg["planet_A_max"]),     # planet A
        (cfg["planet_B_min"], cfg["planet_B_max"]),     # planet B
    ]

C.param

In [ ]:
fitted = [k for k, key in enumerate(C.param) if key not in ('rot.sig', 'rot.beta_1')]
params = [C.param[k] for k in fitted]
x0 = C.get_param(params)
x0 = np.append(x0, [
    cfg["pcyc_init"], cfg["phi_init"], cfg["k_init"],
    cfg["gamma_0_init"], cfg["gamma_1_init"], cfg["delta_0_init"], cfg["delta_1_init"],
])
if FIT_PLANET:
    x0 = np.append(x0, [(cfg["planet_p_min"] + cfg["planet_p_max"]) / 2, cfg["planet_A_min"] * 5, cfg["planet_B_min"] * 5])

result = fmin_l_bfgs_b(mf.negloglike, x0, args=(y_full, C, params, fitted, t_full, series_index, rv_std, FIT_PLANET), bounds=bounds_list)
xbest = result[0]

C.set_param(xbest[:len(params)], params)
C.set_param([0.0], ['rot.beta_1'])

print(params)
print(xbest)

In [ ]:
fit_plot_name = f"run_model_fit_{STAR}.png"
tsmooth, mu, res_rv, res_rhk = mf.plot_fit(
    t_full, y_full, yerr_full, C, xbest, series_index,
    rv_std=rv_std, output_name=fit_plot_name, return_residuals=True, inject_planet=FIT_PLANET,
)

In [ ]:
def log_prior(theta):
    if FIT_PLANET:
        rv_jit, rhk_jit, rot_P0, rot_Q, rot_alpha_0, rot_alpha_1, rot_beta_0, pcyc, phi, k, gamma_0, gamma_1, delta_0, delta_1, planet_p, planet_A, planet_B = theta
    else:
        rv_jit, rhk_jit, rot_P0, rot_Q, rot_alpha_0, rot_alpha_1, rot_beta_0, pcyc, phi, k, gamma_0, gamma_1, delta_0, delta_1 = theta
    phi_wrapped = phi % (2 * np.pi)
    ok = (
        0.0 < rv_jit < rvjit_max and 0.0 < rhk_jit < rhkjit_max
        and cfg["prot_min"] < rot_P0 < cfg["prot_max"] and cfg["Q_min"] < rot_Q < cfg["Q_max"]
        and 0.0 < rot_alpha_0 < alpha_0_max and -alpha_1_max < rot_alpha_1 < alpha_1_max
        and -beta_0_max < rot_beta_0 < beta_0_max
        and cfg["pcyc_min"] < pcyc < cfg["pcyc_max"] and 0.0 < phi_wrapped < 2 * np.pi and -0.4 < k < 0.4
        and 0.0 < gamma_0 < gamma_0_max and 0.0 < gamma_1 < gamma_1_max
        and delta_0_bound[0] < delta_0 < delta_0_bound[1] and delta_1_bound[0] < delta_1 < delta_1_bound[1]
    )
    if FIT_PLANET:
        ok = ok and cfg["planet_p_min"] < planet_p < cfg["planet_p_max"] \
            and cfg["planet_A_min"] < planet_A < cfg["planet_A_max"] \
            and cfg["planet_B_min"] < planet_B < cfg["planet_B_max"]
    return 0.0 if ok else -np.inf

def log_probability(theta, y, C):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + -1 * mf.negloglike(theta, y, C, params, fitted, t_full, series_index, rv_std, FIT_PLANET)[0]

x0_mcmc = xbest.copy()
ndim = len(x0_mcmc)
pos = x0_mcmc + 1e-4 * np.random.randn(ndim * 3, ndim)
nwalkers, ndim = pos.shape

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=(y_full, C))
sampler.run_mcmc(pos, 3000, progress=True)

In [ ]:
labels = params + ['pcyc', 'phi', 'k', 'gamma_0', 'gamma_1', 'delta_0', 'delta_1']
if FIT_PLANET:
    labels += ['planet_p', 'planet_A', 'planet_B']

samples = sampler.get_chain()
fig, axes = plt.subplots(len(labels), figsize=(10, 2 * len(labels)), sharex=True)
for i in range(ndim):
    ax = axes[i]
    ax.plot(samples[:, :, i], "k", alpha=0.3)
    ax.set_xlim(0, len(samples))
    ax.set_ylabel(labels[i])
    ax.yaxis.set_label_coords(-0.1, 0.5)
axes[-1].set_xlabel("step number")

In [ ]:
flat_samples = sampler.get_chain(discard=1500, thin=15, flat=True)
fig = corner.corner(flat_samples, labels=labels, show_titles=True)

In [ ]:
burn = 1500
thin = 1
flat_samples = sampler.get_chain(discard=burn, thin=thin, flat=True)
flat_logp = sampler.get_log_prob(discard=burn, thin=thin, flat=True)
finite_mask = np.isfinite(flat_logp)
flat_samples = flat_samples[finite_mask]
flat_logp = flat_logp[finite_mask]

map_idx = np.argmax(flat_logp)
map_sample = flat_samples[map_idx]
print("MAP log-posterior:", flat_logp[map_idx])
print("MAP sample:", map_sample)

C.set_param(map_sample[:len(params)], params)
C.set_param([0.0], ['rot.beta_1'])
C.set_param([np.var(y_full)], ['rot.sig'])

map_fit_plot_name = f"run_model_map_fit_{STAR}.png"
tsmooth, mu, res_rv, res_rhk = mf.plot_fit(
    t_full, y_full, yerr_full, C, map_sample, series_index,
    rv_std=rv_std, output_name=map_fit_plot_name, return_residuals=True, inject_planet=FIT_PLANET,
)

plt.figure()
plt.title("Residuals")
plt.plot(t_full[series_index[0]], res_rv, ".", label='RV residuals')
plt.legend()
plt.figure()
plt.plot(t_full[series_index[1]], res_rhk, ".", label='RHK residuals')
plt.legend()